# NB_scorecard · De-Identification Compliance Scorecard

> **SYNTHETIC DATA ONLY.** A passing scorecard here is evidence the *pattern* works — it
> is **not** a HIPAA de-identification certification. Real data needs a qualified reviewer.

Automated assertions over the `gold_safe_*` layer that the semantic model consumes:
1. **No direct-identifier patterns** (SSN / phone / email) survive in any string column.
2. **Tokenized columns are tokens** (e.g. every `MRN` starts with `PT-`).
3. **No full date-of-birth** — only `BirthYear` (int) is present.
4. **ZIP is 3-digit** and restricted low-population prefixes are zeroed.
5. **k-anonymity** ≥ k over the quasi-identifier set (BirthYear, Gender, Race, ZIP).
6. **l-diversity / t-closeness** on a sensitive attribute — guards against attribute
   disclosure even when k-anonymity holds *(advisory)*.
7. **Determination is in-date** — the de-id method has a signed, non-expired review-by
   date recorded in the evidence artifact *(advisory)*.

Checks 1–4 are **hard gates** (any failure raises and blocks Gold). The disclosure-risk
metrics (5–7) are **advisory** on synthetic data — reported and persisted, not gated.


In [ ]:
from pyspark.sql import functions as F
import datetime
import sys

# --- make the accelerator's helpers importable (driver-only work, no UDFs) ---
SRC_PATH = "/lakehouse/default/Files/accelerator"
if SRC_PATH not in sys.path:
    sys.path.append(SRC_PATH)
from fabric_phi_deid.audit import get_audit_logger  # noqa: E402
from fabric_phi_deid.privacy_metrics import (  # noqa: E402
    measure_k_anonymity_spark,
    measure_l_diversity,
    measure_t_closeness,
)

log = get_audit_logger()

# --- disclosure-risk thresholds (advisory on synthetic data; tighten for real determinations) ---
K_ANON = 5                       # min equivalence-class size over the quasi-identifiers
L_DIVERSITY = 2                  # min distinct sensitive values per class (attribute-disclosure guard)
T_CLOSENESS = 0.2                # max distribution distance per class (skew guard); smaller = safer
SENSITIVE_ATTR = "Ethnicity"     # sensitive attribute for l-diversity / t-closeness (if present)
DIVERSITY_SAMPLE = 50000         # bound the driver-side pull for the l-div/t-closeness sample

# --- time-limited determination metadata (HHS treats an Expert Determination as expiring) -------
# In PRODUCTION these come from the signed determination on file; here they scaffold the record so
# every scorecard artifact is auditably traceable to a method + review-by date + reviewer.
REVIEW_PERIOD_DAYS = 365
DETERMINATION_METHOD = "safe_harbor"
DETERMINATION_REVIEWER = "UNSIGNED — synthetic demo (no determination on file)"
_now_utc = datetime.datetime.now(datetime.timezone.utc)
DETERMINATION_EXPIRES_UTC = (_now_utc + datetime.timedelta(days=REVIEW_PERIOD_DAYS)).isoformat()
_expires_dt = datetime.datetime.fromisoformat(DETERMINATION_EXPIRES_UTC)
if _expires_dt.tzinfo is None:
    _expires_dt = _expires_dt.replace(tzinfo=datetime.timezone.utc)
DETERMINATION_EXPIRED = _now_utc >= _expires_dt

results = []                     # (check, passed, detail, advisory)
def record(check, passed, detail="", advisory=False):
    results.append((check, bool(passed), detail, bool(advisory)))

GOLD = "gold_safe_"
gold_tables = [t.name for t in spark.catalog.listTables() if t.name.startswith(GOLD)]

# --- fail-fast: an empty gold layer would make every leak check pass VACUOUSLY (false green) ---
if not gold_tables:
    raise RuntimeError(
        f"No {GOLD}* tables found — nothing to score. Run 03b_gold_safe first "
        "(with lh_raw pinned as the default lakehouse)."
    )
if f"{GOLD}dim_patient" not in gold_tables:
    raise RuntimeError(f"Missing {GOLD}dim_patient — 03b_gold_safe did not complete. Re-run it.")

patient = spark.table(f"{GOLD}dim_patient")
patient_rows = patient.count()
if patient_rows == 0:
    raise RuntimeError(f"{GOLD}dim_patient is EMPTY — scorecard cannot certify an empty table.")

print(f"Scoring {len(gold_tables)} tables ({patient_rows:,} patients): {gold_tables}")


In [ ]:
# --- 1. No SSN / phone / email patterns in ANY string column of ANY gold_safe table ---
SSN   = r"^\d{3}-\d{2}-\d{4}$"
PHONE = r"^(\+?1[-. ]?)?\(?\d{3}\)?[-. ]?\d{3}[-. ]?\d{4}$"
EMAIL = r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}"
leaks = []
for tname in gold_tables:
    df = spark.table(tname)
    string_cols = [c for c, t in df.dtypes if t == "string"]
    for c in string_cols:
        hit = df.where(
            F.col(c).rlike(SSN) | F.col(c).rlike(PHONE) | F.col(c).rlike(EMAIL)
        ).limit(1).count()
        if hit:
            leaks.append(f"{tname}.{c}")
record("No SSN/phone/email patterns", len(leaks) == 0, f"leaky cols: {leaks}")

In [ ]:
# --- 2. MRN is fully tokenized (every value starts with PT-) ---
bad_mrn = patient.where(~F.col("MRN").startswith("PT-")).limit(1).count()
record("MRN is tokenized (PT- prefix)", bad_mrn == 0)

# --- 3. No raw date-of-birth column; BirthYear present and 4-digit ---
cols = set(patient.columns)
record("No DateOfBirth column in Gold", "DateOfBirth" not in cols)
record("BirthYear present", "BirthYear" in cols)
if "BirthYear" in cols:
    bad_year = patient.where((F.col("BirthYear") < 1900) | (F.col("BirthYear") > 2100)).limit(1).count()
    record("BirthYear is a plausible 4-digit year", bad_year == 0)

In [ ]:
# --- 4. ZIP is at most 3 digits everywhere it appears (patient geography) ---
bad_zip = patient.where(F.length(F.col("ZIP")) > 3).limit(1).count()
record("Patient ZIP is <= 3 digits", bad_zip == 0)

# --- 5. k-anonymity over quasi-identifiers (distributed over the FULL table) ---
quasi = [c for c in ["BirthYear", "Gender", "Race", "ZIP"] if c in patient.columns]
k_report = measure_k_anonymity_spark(patient.select(*quasi), quasi, k=K_ANON)
record(f"k-anonymity >= {K_ANON} on {quasi}", k_report.passes,
       f"k={k_report.k}; {k_report.violating_records} rows in {k_report.violating_classes} "
       "small groups (expected on synthetic data; tune generalization)",
       advisory=True)

# --- 6/7. l-diversity + t-closeness of the sensitive attribute within QI classes (advisory) ---
# These need per-record values, so pull only [QIs + sensitive] up to a bounded sample to the driver.
if SENSITIVE_ATTR in patient.columns and quasi:
    qi_for_div = [c for c in quasi if c != SENSITIVE_ATTR]
    sample = patient.select(*qi_for_div, SENSITIVE_ATTR).limit(DIVERSITY_SAMPLE).collect()
    records_sample = [r.asDict() for r in sample]
    note = f"sample<= {DIVERSITY_SAMPLE:,}; advisory" if patient_rows > DIVERSITY_SAMPLE else "advisory"

    l_report = measure_l_diversity(records_sample, qi_for_div, SENSITIVE_ATTR, l=L_DIVERSITY)
    record(f"l-diversity >= {L_DIVERSITY} for '{SENSITIVE_ATTR}'", l_report.passes,
           f"l={l_report.l}; {l_report.violating_records} rows in "
           f"{l_report.violating_classes} classes ({note})", advisory=True)

    t_report = measure_t_closeness(records_sample, qi_for_div, SENSITIVE_ATTR, t=T_CLOSENESS)
    record(f"t-closeness <= {T_CLOSENESS} for '{SENSITIVE_ATTR}'", t_report.passes,
           f"t={t_report.t:.3f}; {t_report.violating_records} rows in "
           f"{t_report.violating_classes} classes ({note})", advisory=True)
else:
    record(f"l-diversity / t-closeness for '{SENSITIVE_ATTR}'", True,
           f"skipped — '{SENSITIVE_ATTR}' not in dim_patient", advisory=True)

# --- 8. Determination is a signed, in-date method (governance, not a data leak) ---
record("De-id determination not expired", not DETERMINATION_EXPIRED,
       f"method={DETERMINATION_METHOD}, review by {DETERMINATION_EXPIRES_UTC}", advisory=True)


In [ ]:
# --- Report + hard gate + PHI-free evidence artifact ---
print(f"{'CHECK':45s} {'RESULT':8s} DETAIL")
print("-" * 100)
for check, passed, detail, advisory in results:
    result = ("PASS" if passed else "FAIL") + (" *" if advisory else "")
    print(f"{check:45s} {result:8s} {detail}")
print("\n* advisory (k-anonymity / l-diversity / t-closeness / determination) — reported, not gated on synthetic data.")

# Direct-identifier / structural checks are hard gates; disclosure-risk metrics are advisory here.
hard_fail = [c for c, p, _, adv in results if not p and not adv]

# Persist a metadata-only scorecard record (no PHI, no data values) as an audit artifact.
import json, uuid, datetime, os
scorecard_id = str(uuid.uuid4())
record_out = {
    "scorecard_id": scorecard_id,
    "utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "gold_tables_scored": gold_tables,
    "patient_rows": patient_rows,
    "thresholds": {
        "k_anonymity": K_ANON,
        "l_diversity": L_DIVERSITY,
        "t_closeness": T_CLOSENESS,
        "sensitive_attribute": SENSITIVE_ATTR,
    },
    "determination": {
        "method": DETERMINATION_METHOD,
        "expires_utc": DETERMINATION_EXPIRES_UTC,
        "reviewer": DETERMINATION_REVIEWER,
        "expired": DETERMINATION_EXPIRED,
    },
    "checks": [
        {"check": c, "passed": p, "detail": d, "advisory": adv}
        for c, p, d, adv in results
    ],
    "hard_gate_passed": len(hard_fail) == 0,
}
_audit_dir = "/lakehouse/default/Files/audit"
os.makedirs(_audit_dir, exist_ok=True)
_path = f"{_audit_dir}/scorecard_{scorecard_id}.json"
with open(_path, "w", encoding="utf-8") as fh:
    json.dump(record_out, fh, indent=2)

if DETERMINATION_EXPIRED:
    log.warning(f"scorecard {scorecard_id}: de-id determination EXPIRED ({DETERMINATION_EXPIRES_UTC}) — re-review before real PHI.")

if hard_fail:
    log.error(f"scorecard {scorecard_id} FAILED hard gate: {hard_fail}")
    raise AssertionError(f"De-identification scorecard FAILED: {hard_fail}  (record: {_path})")

log.info(f"scorecard {scorecard_id} PASSED — {len(results)} checks, artifact {_path}")
print(f"\nScorecard PASSED (disclosure-risk metrics are advisory on synthetic data).")
print(f"Evidence artifact written: {_path}")
